# OneVoice V2 — Safety phrase review

Tạo bản CSV review trên Drive. Safety officer/reviewer phải xác nhận từng phrase trước khi sinh audio. Không có cell nào tự đặt `approved`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys

GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
REPO = Path('/content/OneVoice')
WORK_ROOT = Path('/content/drive/MyDrive/OneVoice')
REVIEW_CSV = WORK_ROOT / 'review/safety_fast_path_review.csv'

if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', GITHUB_REPO, str(REPO)], check=True)
os.chdir(REPO)
print('Review file:', REVIEW_CSV)


In [ ]:
if REVIEW_CSV.is_file():
    print('Review CSV already exists; preserving reviewer edits.')
else:
    subprocess.run([
        sys.executable, 'scripts/prepare_safety_review.py',
        '--input', 'data/onevoice_construction_v2/safety_fast_path.csv',
        '--output', str(REVIEW_CSV),
    ], check=True)

import csv
from collections import Counter
with REVIEW_CSV.open(encoding='utf-8-sig', newline='') as handle:
    rows = list(csv.DictReader(handle))
print('Rows:', len(rows), '| statuses:', Counter(row.get('review_status', '') for row in rows))
print('Open this CSV in Google Sheets. For every fixed_translation_candidate=True row, set review_status to approved/rejected; approved rows also require reviewer and reviewed_at (YYYY-MM-DD).')


In [ ]:
# Run only after the reviewed CSV has been saved back to the same Drive path.
subprocess.run([sys.executable, 'scripts/validate_safety_review.py', '--input', str(REVIEW_CSV)], check=True)
print('Review gate passed. Continue with colab_safety_audio_v2.ipynb.')
